# Iris chatbot with Ollama + pandas tools (self-hosted)

**Two-model pipeline (cheap + capable):**

| Role | Model | Why |
|------|--------|-----|
| **Agent** (tool calling) | `granite3-dense:2b` (~1.6 GB) | Supports tools; strong enough for pandas queries |
| **Formatter** (friendly text) | `granite3-moe:1b` (~822 MB) | Smallest Granite; fast summaries |

Self-hosted “cost” ≈ disk + RAM + tokens/sec — smaller models are cheaper. **Avoid `granite3-dense:8b` for formatting** (4× heavier, no benefit here).

**One-time pull for the formatter:**

```bash
ollama pull granite3-moe:1b
```

If tool calls fail on Granite, set `AGENT_MODEL = "llama3.2"`.

**Docker:** see [README.md](README.md) — `docker compose up -d ollama && docker compose up pull-models`, then `docker compose --profile cli run --rm chatbot "your question"`.

In [8]:
# Docker: OLLAMA_HOST=http://ollama:11434 — see README.md
from app.iris_agent import df, ask_iris_friendly

df.head()

,sepal_length,sepal_width,petal_length,petal_width,target,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


In [9]:
ALLOWED_COLUMNS = set(df.columns)
ALLOWED_AGG = {"mean", "max", "min", "sum", "count"}


def _truncate(result: pd.DataFrame | pd.Series) -> str:
    if isinstance(result, pd.Series):
        return result.to_string()
    if len(result) > MAX_ROWS:
        return (
            result.head(MAX_ROWS).to_string(index=False)
            + f"\n... ({len(result) - MAX_ROWS} more rows)"
        )
    return result.to_string(index=False)


def dataframe_info() -> str:
    """Return Iris dataset schema, row count, and summary statistics.

    Returns:
        str: Column names, dtypes, species list, and describe() output.
    """
    species = ", ".join(sorted(df["species"].unique()))
    return (
        f"rows={len(df)}, columns={list(df.columns)}\n"
        f"species: {species}\n\n"
        f"{df.describe().to_string()}\n\n"
        f"dtypes:\n{df.dtypes.to_string()}"
    )


def query_dataframe(expression: str) -> str:
    """Filter Iris rows with pandas DataFrame.query() syntax.

    Args:
        expression: Query string, e.g. "species == 'setosa' and sepal_width < 3.5".

    Returns:
        str: Matching rows as a table (capped at MAX_ROWS).
    """
    subset = df.query(expression, engine="python")
    return _truncate(subset)


def groupby_aggregate(
    group_by: str,
    column: str,
    agg: Literal["mean", "max", "min", "sum", "count"],
) -> str:
    """Aggregate a numeric column grouped by a column (usually species).

    Args:
        group_by: Column to group by, e.g. "species".
        column: Numeric column to aggregate, e.g. "petal_length".
        agg: One of mean, max, min, sum, count.

    Returns:
        str: Aggregation result table.
    """
    if group_by not in ALLOWED_COLUMNS:
        raise ValueError(f"Unknown group_by: {group_by}")
    if column not in ALLOWED_COLUMNS:
        raise ValueError(f"Unknown column: {column}")
    if agg not in ALLOWED_AGG:
        raise ValueError(f"agg must be one of {ALLOWED_AGG}")

    result = df.groupby(group_by, as_index=False)[column].agg(agg)
    return _truncate(result)


def value_counts(column: str) -> str:
    """Count distinct values in a column.

    Args:
        column: Column name, e.g. "species".

    Returns:
        str: Value counts table.
    """
    if column not in ALLOWED_COLUMNS:
        raise ValueError(f"Unknown column: {column}")
    return df[column].value_counts().to_string()


PANDAS_TOOLS = [dataframe_info, query_dataframe, groupby_aggregate, value_counts]
TOOL_BY_NAME = {fn.__name__: fn for fn in PANDAS_TOOLS}

In [10]:
def _run_tool(name: str, arguments: dict) -> str:
    fn = TOOL_BY_NAME.get(name)
    if fn is None:
        return f"Unknown tool: {name}"
    try:
        return fn(**arguments)
    except Exception as exc:
        return f"Tool error ({name}): {exc}"


def ask_iris(question: str, verbose: bool = False) -> dict:
    """Run the agent: pandas tools + short technical draft from AGENT_MODEL."""
    messages = [
        {
            "role": "system",
            "content": (
                "You answer questions about the Iris flower dataset. "
                "Always use the pandas tools to look up data before answering. "
                "Reply briefly and factually. Do not invent numbers. "
                "Columns: sepal_length, sepal_width, petal_length, petal_width, "
                "species (setosa, versicolor, virginica)."
            ),
        },
        {"role": "user", "content": question},
    ]
    tool_trace: list[str] = []

    for _ in range(MAX_TOOL_STEPS):
        response = chat(model=AGENT_MODEL, messages=messages, tools=PANDAS_TOOLS)
        assistant = response.message
        messages.append(assistant.model_dump(exclude_none=True))

        if not assistant.tool_calls:
            return {
                "question": question,
                "draft": assistant.content or "",
                "tool_trace": tool_trace,
            }

        for call in assistant.tool_calls:
            name = call.function.name
            args = call.function.arguments
            if isinstance(args, str):
                args = json.loads(args)
            result = _run_tool(name, args)
            tool_trace.append(f"{name}({args})\n{result}")
            if verbose:
                print(f"[tool] {name}({args})\n{result}\n")
            messages.append({
                "role": "tool",
                "tool_name": name,
                "content": result,
            })

    return {
        "question": question,
        "draft": "Stopped: too many tool steps.",
        "tool_trace": tool_trace,
    }


def format_friendly(result: dict) -> str:
    """Rewrite agent output into plain language using the small FORMATTER_MODEL."""
    evidence = "\n\n---\n\n".join(result["tool_trace"]) or "(no tool output)"
    prompt = f"""Rewrite this data assistant reply for a non-technical user.

Rules:
- Use 2–4 short sentences, friendly and clear.
- Include the key numbers and species names from the evidence.
- Do not mention tools, pandas, SQL, or models.
- If the evidence is insufficient, say what is missing.

User question: {result['question']}

Technical draft: {result['draft']}

Data evidence:
{evidence}
"""

    response = chat(
        model=FORMATTER_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.3, "num_predict": 200},
    )
    return response.message.content or ""


def ask_iris_friendly(question: str, verbose: bool = False) -> str:
    """Full pipeline: agent (tools) → formatter (readable answer)."""
    result = ask_iris(question, verbose=verbose)
    return format_friendly(result)

In [11]:
# Friendly answer (agent + formatter models)
print(ask_iris_friendly("Which species has the largest average petal length?", verbose=True))

ResponseError: model 'granite3-moe:1b' not found (status code: 404)

In [ ]:
ask_iris("How many setosa flowers have sepal width under 3.5 cm?", verbose=True)

In [ ]:
!ollama pull granite3-moe:1b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 6bd48eb2b42c:   0% ▕                  ▏  39 KB/821 MB                  pulling manifest 
pulling 6bd48eb2b42c:   0% ▕                  ▏ 955 KB/821 MB                  pulling manifest 
pulling 6bd48eb2b42c:   0% ▕                  ▏ 1.6 MB/821 MB                  pulling manifest 
pulling 6bd48eb2b42c:   0% ▕                  ▏ 2.1 MB/821 MB                  pulling manifest 
pulling 6bd48eb2b42c:   0% ▕                  ▏ 2.4 MB/821 MB                  pulling manifest 
pulling 6bd48eb2b42c:   0% ▕                  ▏ 3.4 

In [1]:
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 141 KB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 444 KB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 1.3 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 2.8 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 2.8 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 3.3 MB/2.0 GB                  pulling manifest 
pulling dde5

In [ ]:
pip install torch transformers pillow pytesseract scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install ollama

^C
Note: you may need to restart the kernel to use updated packages.


In [2]:
!ollama pull gemma3:4b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling aeda25e63ebd:   0% ▕                  ▏ 413 KB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   0% ▕                  ▏ 1.3 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   0% ▕                  ▏ 2.3 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   0% ▕                  ▏ 2.8 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   0% ▕                  ▏ 3.2 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   0% ▕                  ▏ 4.4 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   0% ▕                  ▏ 4.

In [6]:
!ollama pull granite3-dense:2b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 63dd4fe4571a:   0% ▕                  ▏ 572 KB/1.6 GB                  pulling manifest 
pulling 63dd4fe4571a:   0% ▕                  ▏ 935 KB/1.6 GB                  pulling manifest 
pulling 63dd4fe4571a:   0% ▕                  ▏ 1.4 MB/1.6 GB                  pulling manifest 
pulling 63dd4fe4571a:   0% ▕                  ▏ 2.7 MB/1.6 GB                  pulling manifest 
pulling 63dd4fe4571a:   0% ▕                  ▏ 3.7 MB/1.6 GB                  pulling manifest 
pulling 63dd4fe4571a:   0% ▕                  ▏ 4.0 MB/1.6 GB                  pulling man

In [3]:
from sklearn.datasets import load_iris

# Load the dataset
sample_data = load_iris(as_frame=True)

In [ ]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model='granite3-dense:2b', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])
# or access fields directly from the response object
print(response.message.content)

The sky appears blue due to a process called Rayleigh scattering. As sunlight reaches Earth's atmosphere, it is scattered in all directions by the tiny molecules and particles in the air. Blue light is scattered more than other colors because it travels in shorter, smaller waves. This scattered blue light is what we see when we look up at the sky.
The sky appears blue due to a process called Rayleigh scattering. As sunlight reaches Earth's atmosphere, it is scattered in all directions by the tiny molecules and particles in the air. Blue light is scattered more than other colors because it travels in shorter, smaller waves. This scattered blue light is what we see when we look up at the sky.
